# SDE PSD difference-from-glossy notebook using the original diffuser

This notebook learns the residual `glossy - diffuse` from glossy conditioning using `SDEBackbone.UNetWithTransformer` and the original `Diffuser.py`. The diffuser file is not modified.


In [ ]:
from __future__ import annotations

import math
import os
import random
import re
import sys
from collections import OrderedDict
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def is_specdiff_root(path: Path) -> bool:
    return (path / 'Run_Training.py').exists() and (path / 'SDEBackbone.py').exists() and (path / 'Diffuser.py').exists()


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend([
            base,
            base / 'SpecDiff',
            base / 'Specular-Highlights' / 'SpecDiff',
        ])

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if is_specdiff_root(candidate):
            return candidate
    raise FileNotFoundError('Could not find the SpecDiff repo root from the current working directory.')


def find_psd_root(root: Path) -> Path:
    env_candidates = [
        os.environ.get('SPECDIFF_PSD_ROOT'),
        os.environ.get('PSD_DATASET_ROOT'),
    ]
    candidate_paths = [Path(value).expanduser() for value in env_candidates if value]
    workspace_root = root.parent.parent
    candidate_paths.extend([
        Path('/share/lcn_projects/z0058vfs/project_hl.PSD_Dataset'),
        Path('/share/lcn_projects/z0058vfs/project_hl/PSD_Dataset'),
        Path('/share/lcn_projects/z0058vfs/project_hl/PSD_Dataset/PSD_Dataset'),
        workspace_root / 'PSD_Dataset' / 'PSD_Dataset',
        workspace_root / 'PSD_Dataset',
        root.parent / 'PSD_Dataset' / 'PSD_Dataset',
        root.parent / 'PSD_Dataset',
    ])

    seen = set()
    for candidate in candidate_paths:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        required_dirs = [
            candidate / 'PSD_Train' / 'PSD_Train_specular',
            candidate / 'PSD_Train' / 'PSD_Train_diffuse',
            candidate / 'PSD_val' / 'PSD_val_specular',
            candidate / 'PSD_val' / 'PSD_val_diffuse',
        ]
        if all(path.exists() for path in required_dirs):
            return candidate
    raise FileNotFoundError('Could not find the PSD dataset root. Set SPECDIFF_PSD_ROOT or PSD_DATASET_ROOT.')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import SDEBackbone
import Diffuser as diff


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR = ROOT / 'models' / '32'
CHECKPOINTS_DIR = ROOT / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'repo root: {ROOT}')
print(f'device: {DEVICE}')


In [ ]:
SEED = 42
IMAGE_SIZE = 32
DEPTH = 4
BATCH_SIZE = 10
NOISE_STEPS = 200
EPOCHS = 2000
LR = 1e-4
FINAL_LR = 1e-5
WARMUP_EPOCHS = 100
EMA_DECAY = 0.9999

NOTEBOOK_GROUP = 'sde_psd_normal_diffuser'
TASK_MODE = 'difference'
CONDITION_SOURCE = 'glossy'
TARGET_SOURCE = 'difference'
EVAL_EVERY = 200
CHECKPOINT_EVERY = 500
VAL_BATCHES = 4
VAL_CASE_INDEX = 0
EVAL_SEED = 123
RESTORE_START_STEP = 40

PSD_ROOT = find_psd_root(ROOT)
TRAIN_GLOSSY_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_specular'
TRAIN_DIFFUSE_DIR = PSD_ROOT / 'PSD_Train' / 'PSD_Train_diffuse'
VAL_GLOSSY_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_specular'
VAL_DIFFUSE_DIR = PSD_ROOT / 'PSD_val' / 'PSD_val_diffuse'

SAVE_STEM = 'SDE_PSD_DifferenceFromGlossy_NormalDiffuser_Flex32'
TRAINER_SAVE_PATH = CHECKPOINTS_DIR / f'{SAVE_STEM}_checkpoint'
FINAL_MODEL_PATH = MODELS_DIR / f'{SAVE_STEM}.pth'

print(f'PSD root: {PSD_ROOT}')
print(f'task mode: {TASK_MODE}')
print(f'condition source: {CONDITION_SOURCE}')
print(f'target source: {TARGET_SOURCE}')
print(f'final model path: {FINAL_MODEL_PATH}')


In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
PIL_BILINEAR = getattr(Image, 'Resampling', Image).BILINEAR


def normalize_image_key(name: str) -> str:
    stem = Path(name).stem.lower()
    stem = stem.replace('specular', '').replace('glossy', '').replace('diffuse', '')
    return re.sub(r'[^a-z0-9]+', '', stem)


def pair_image_paths(glossy_dir: Path, diffuse_dir: Path):
    glossy_candidates = [path for path in glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS]
    diffuse_candidates = [path for path in diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS]

    glossy_files = {path.name: path for path in glossy_candidates}
    diffuse_files = {path.name: path for path in diffuse_candidates}
    exact_names = sorted(set(glossy_files) & set(diffuse_files))
    if exact_names:
        return [(glossy_files[name], diffuse_files[name], name) for name in exact_names]

    glossy_by_key = {}
    for path in glossy_candidates:
        key = normalize_image_key(path.name)
        if key in glossy_by_key:
            raise RuntimeError(f'Duplicate glossy normalized key {key!r} in {glossy_dir}')
        glossy_by_key[key] = path

    diffuse_by_key = {}
    for path in diffuse_candidates:
        key = normalize_image_key(path.name)
        if key in diffuse_by_key:
            raise RuntimeError(f'Duplicate diffuse normalized key {key!r} in {diffuse_dir}')
        diffuse_by_key[key] = path

    common_keys = sorted(set(glossy_by_key) & set(diffuse_by_key))
    if not common_keys:
        raise RuntimeError(f'No paired PSD samples found in {glossy_dir} and {diffuse_dir}')

    return [(glossy_by_key[key], diffuse_by_key[key], glossy_by_key[key].name) for key in common_keys]


def load_rgb_tensor(path: Path, image_size: int) -> torch.Tensor:
    image = Image.open(path).convert('RGB')
    image = image.resize((image_size, image_size), resample=PIL_BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous()


def build_condition_and_target(glossy: torch.Tensor, diffuse: torch.Tensor, task_mode: str):
    if task_mode == 'direct_diffuse':
        return glossy, diffuse, diffuse
    if task_mode == 'difference':
        return glossy, glossy - diffuse, diffuse
    if task_mode == 'diffuse_prior':
        return diffuse, diffuse, diffuse
    raise ValueError(f'Unknown task mode: {task_mode}')


class PSDTaskDataset(Dataset):
    def __init__(self, glossy_dir: Path, diffuse_dir: Path, image_size: int, task_mode: str):
        self.glossy_dir = Path(glossy_dir)
        self.diffuse_dir = Path(diffuse_dir)
        self.image_size = int(image_size)
        self.task_mode = task_mode

        if not self.glossy_dir.exists():
            raise FileNotFoundError(f'Missing glossy directory: {self.glossy_dir}')
        if not self.diffuse_dir.exists():
            raise FileNotFoundError(f'Missing diffuse directory: {self.diffuse_dir}')

        self.samples = pair_image_paths(self.glossy_dir, self.diffuse_dir)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        glossy_path, diffuse_path, name = self.samples[idx]
        glossy = load_rgb_tensor(glossy_path, self.image_size)
        diffuse = load_rgb_tensor(diffuse_path, self.image_size)
        condition, target, true_diffuse = build_condition_and_target(glossy, diffuse, self.task_mode)
        meta = {
            'name': name,
            'glossy_path': str(glossy_path),
            'diffuse_path': str(diffuse_path),
        }
        return condition, target, true_diffuse, meta


train_dataset = PSDTaskDataset(TRAIN_GLOSSY_DIR, TRAIN_DIFFUSE_DIR, image_size=IMAGE_SIZE, task_mode=TASK_MODE)
val_dataset = PSDTaskDataset(VAL_GLOSSY_DIR, VAL_DIFFUSE_DIR, image_size=IMAGE_SIZE, task_mode=TASK_MODE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

fixed_val_condition, fixed_val_target, fixed_val_diffuse, fixed_val_meta = val_dataset[VAL_CASE_INDEX]
fixed_val_condition = fixed_val_condition.unsqueeze(0).to(DEVICE)
fixed_val_target = fixed_val_target.unsqueeze(0).to(DEVICE)
fixed_val_diffuse = fixed_val_diffuse.unsqueeze(0).to(DEVICE)

fixed_eval_generator = torch.Generator()
fixed_eval_generator.manual_seed(EVAL_SEED)
fixed_eval_noise = torch.randn(fixed_val_target.shape, generator=fixed_eval_generator, dtype=fixed_val_target.dtype).to(DEVICE)

print(f'train dataset size: {len(train_dataset)}')
print(f'val dataset size: {len(val_dataset)}')
print(f'train batches per epoch: {len(train_loader)}')
print(f'fixed val case: {fixed_val_meta["name"]}')
print(f'fixed condition shape: {tuple(fixed_val_condition.shape)}')
print(f'fixed target shape: {tuple(fixed_val_target.shape)}')
print(f'fixed diffuse shape: {tuple(fixed_val_diffuse.shape)}')


In [ ]:
def get_cosine_lambda(initial_lr: float, final_lr: float, epochs: int, warmup_epoch: int):
    def cosine_lambda(epoch_idx: int) -> float:
        if epoch_idx < warmup_epoch:
            return epoch_idx / max(warmup_epoch, 1)
        cosine = (math.cos((epoch_idx - warmup_epoch) / max(epochs - warmup_epoch, 1) * math.pi) + 1.0) / 2.0
        return 1.0 - (1.0 - cosine) * (1.0 - final_lr / initial_lr)

    return cosine_lambda


def checkpoint_save(model, optimizer, loss: float, epoch: int, save_path: Path):
    model_dir = Path(f'{save_path}_epoch_{epoch}_loss_{loss:.4f}')
    model_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'epoch': int(epoch),
            'loss': float(loss),
            'notebook_group': NOTEBOOK_GROUP,
            'task_mode': TASK_MODE,
            'condition_source': CONDITION_SOURCE,
            'target_source': TARGET_SOURCE,
            'restore_prior_source': 'glossy',
            'restore_start_step': RESTORE_START_STEP,
            'image_size': IMAGE_SIZE,
            'noise_steps': NOISE_STEPS,
            'depth': DEPTH,
            'diffuser_module': 'Diffuser',
            'sampling_mode': 'ddpm_safe_e_with_restore_eval',
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },
        model_dir / 'model.pth',
    )


def update_ema(ema_model, model, decay: float = 0.9999):
    ema_params = OrderedDict(ema_model.named_parameters())
    model_params = OrderedDict(model.named_parameters())
    for name, param in model_params.items():
        if name in ema_params:
            ema_params[name].data.mul_(decay).add_(param.data, alpha=1.0 - decay)


def ddpm_sample_safe(
    model: torch.nn.Module,
    diffuser: diff.Diffuser,
    condition: torch.Tensor,
    initial_noise: torch.Tensor | None = None,
    show_progress: bool = False,
) -> torch.Tensor:
    with torch.no_grad():
        if initial_noise is None:
            x_t = torch.randn_like(condition)
        else:
            x_t = initial_noise.to(condition.device, dtype=condition.dtype).clone()

        t_now = torch.full((x_t.shape[0],), diffuser.steps - 1, device=condition.device, dtype=torch.long)
        iterator = range(diffuser.steps - 1)
        if show_progress:
            iterator = tqdm(iterator, desc='DDPM sampling', leave=False)

        for _ in iterator:
            t_pre = torch.clamp(t_now - 1, min=0)
            predicted_noise = model(x_t, t_now, condition)
            x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            t_now = t_pre

        return x_t


def ddpm_restore_safe(
    model: torch.nn.Module,
    diffuser: diff.Diffuser,
    condition: torch.Tensor,
    restore_prior: torch.Tensor,
    start_step: int,
    initial_noise: torch.Tensor | None = None,
    show_progress: bool = False,
) -> torch.Tensor:
    with torch.no_grad():
        start_step = int(max(1, min(start_step, diffuser.steps - 1)))
        if initial_noise is None:
            noise = torch.randn_like(restore_prior)
        else:
            noise = initial_noise.to(restore_prior.device, dtype=restore_prior.dtype).clone()
        t_now = torch.full((restore_prior.shape[0],), start_step, device=restore_prior.device, dtype=torch.long)
        x_t = diffuser.forward_diffusion(restore_prior, t_now, noise)

        iterator = range(start_step)
        if show_progress:
            iterator = tqdm(iterator, desc='DDPM restore', leave=False)

        for _ in iterator:
            t_pre = torch.clamp(t_now - 1, min=0)
            predicted_noise = model(x_t, t_now, condition)
            x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            t_now = t_pre

        return x_t


def prediction_to_diffuse(prediction: torch.Tensor, condition: torch.Tensor, task_mode: str) -> torch.Tensor:
    if task_mode == 'difference':
        return (condition - prediction).clamp(0.0, 1.0)
    return prediction.clamp(0.0, 1.0)


def task_labels(task_mode: str):
    if task_mode == 'difference':
        return 'Difference', 'Glossy condition'
    if task_mode == 'direct_diffuse':
        return 'Diffuse', 'Glossy condition'
    if task_mode == 'diffuse_prior':
        return 'Diffuse', 'Diffuse condition'
    raise ValueError(f'Unknown task mode: {task_mode}')


def normalize_rgb_tensor(rgb: torch.Tensor):
    low = float(rgb.min())
    high = float(rgb.max())
    if abs(high - low) < 1e-8:
        high = low + 1e-8
    return ((rgb - low) / (high - low)).clamp(0.0, 1.0)


def channel_limits(channel: torch.Tensor):
    low = float(channel.min())
    high = float(channel.max())
    if abs(high - low) < 1e-8:
        high = low + 1e-8
    return low, high


def metric_bundle(prediction: torch.Tensor, target: torch.Tensor, true_diffuse: torch.Tensor, condition: torch.Tensor) -> dict:
    predicted_diffuse = prediction_to_diffuse(prediction, condition, TASK_MODE)
    target_mse = float(F.mse_loss(prediction, target).item())
    target_mae = float((prediction - target).abs().mean().item())
    diffuse_mse = float(F.mse_loss(predicted_diffuse, true_diffuse).item())
    diffuse_mae = float((predicted_diffuse - true_diffuse).abs().mean().item())
    diffuse_psnr = float((10.0 * torch.log10(1.0 / F.mse_loss(predicted_diffuse, true_diffuse).clamp_min(1e-10))).item())
    return {
        'prediction': prediction,
        'predicted_diffuse': predicted_diffuse,
        'target_mse': target_mse,
        'target_mae': target_mae,
        'diffuse_mse': diffuse_mse,
        'diffuse_mae': diffuse_mae,
        'diffuse_psnr': diffuse_psnr,
    }


def plot_prediction(
    condition_tensor: torch.Tensor,
    target_tensor: torch.Tensor,
    pure_prediction_tensor: torch.Tensor,
    restore_prediction_tensor: torch.Tensor,
    true_diffuse_tensor: torch.Tensor,
    epoch: int,
):
    condition = condition_tensor.detach().cpu().squeeze(0)
    target = target_tensor.detach().cpu().squeeze(0)
    pure_prediction = pure_prediction_tensor.detach().cpu().squeeze(0)
    restore_prediction = restore_prediction_tensor.detach().cpu().squeeze(0)
    true_diffuse = true_diffuse_tensor.detach().cpu().squeeze(0)

    pure_pred_diffuse = prediction_to_diffuse(pure_prediction, condition, TASK_MODE)
    restore_pred_diffuse = prediction_to_diffuse(restore_prediction, condition, TASK_MODE)
    target_label, condition_label = task_labels(TASK_MODE)

    if TASK_MODE == 'difference':
        target_rgb = normalize_rgb_tensor(target)
        pure_rgb = normalize_rgb_tensor(pure_prediction)
        restore_rgb = normalize_rgb_tensor(restore_prediction)
    else:
        target_rgb = target.clamp(0.0, 1.0)
        pure_rgb = pure_prediction.clamp(0.0, 1.0)
        restore_rgb = restore_prediction.clamp(0.0, 1.0)

    fig, axes = plt.subplots(5, 5, figsize=(20, 18))
    for channel_idx, channel_name in enumerate(['Red', 'Green', 'Blue']):
        target_low, target_high = channel_limits(target[channel_idx])
        pure_low, pure_high = channel_limits(pure_prediction[channel_idx])
        restore_low, restore_high = channel_limits(restore_prediction[channel_idx])

        axes[channel_idx, 0].imshow(target[channel_idx], cmap='coolwarm', vmin=target_low, vmax=target_high)
        axes[channel_idx, 0].set_title(f'{channel_name} {target_label.lower()} target\nmin={target_low:.3f}, max={target_high:.3f}')
        axes[channel_idx, 1].imshow(pure_prediction[channel_idx], cmap='coolwarm', vmin=pure_low, vmax=pure_high)
        axes[channel_idx, 1].set_title(f'{channel_name} pure-noise pred\nmin={pure_low:.3f}, max={pure_high:.3f}')
        axes[channel_idx, 2].imshow((target[channel_idx] - pure_prediction[channel_idx]).abs(), cmap='magma')
        axes[channel_idx, 2].set_title(f'{channel_name} pure abs error')
        axes[channel_idx, 3].imshow(restore_prediction[channel_idx], cmap='coolwarm', vmin=restore_low, vmax=restore_high)
        axes[channel_idx, 3].set_title(f'{channel_name} restore pred\nmin={restore_low:.3f}, max={restore_high:.3f}')
        axes[channel_idx, 4].imshow((target[channel_idx] - restore_prediction[channel_idx]).abs(), cmap='magma')
        axes[channel_idx, 4].set_title(f'{channel_name} restore abs error')

    axes[3, 0].imshow(target_rgb.permute(1, 2, 0).numpy())
    axes[3, 0].set_title(f'RGB {target_label.lower()} target')
    axes[3, 1].imshow(pure_rgb.permute(1, 2, 0).numpy())
    axes[3, 1].set_title('RGB pure-noise pred')
    axes[3, 2].imshow((target - pure_prediction).abs().mean(dim=0).numpy(), cmap='magma')
    axes[3, 2].set_title('RGB pure mean abs error')
    axes[3, 3].imshow(restore_rgb.permute(1, 2, 0).numpy())
    axes[3, 3].set_title('RGB restore pred')
    axes[3, 4].imshow((target - restore_prediction).abs().mean(dim=0).numpy(), cmap='magma')
    axes[3, 4].set_title('RGB restore mean abs error')

    axes[4, 0].imshow(condition.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 0].set_title(condition_label)
    axes[4, 1].imshow(true_diffuse.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 1].set_title('True diffuse')
    axes[4, 2].imshow(pure_pred_diffuse.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 2].set_title('Pure-noise diffuse')
    axes[4, 3].imshow(restore_pred_diffuse.permute(1, 2, 0).numpy().clip(0.0, 1.0))
    axes[4, 3].set_title('Restore diffuse')
    axes[4, 4].imshow((restore_pred_diffuse - true_diffuse).abs().mean(dim=0).numpy(), cmap='magma')
    axes[4, 4].set_title('Restore diffuse abs error')

    for ax in axes.ravel():
        ax.axis('off')
    fig.suptitle(f'Fixed validation inference at epoch {epoch}', fontsize=18)
    plt.tight_layout()
    plt.show()


model = SDEBackbone.UNetWithTransformer(noise_steps=NOISE_STEPS, size=IMAGE_SIZE, depth=DEPTH, conditioning_channels=3).to(DEVICE)
ema_model = deepcopy(model).to(DEVICE)
ema_model.load_state_dict(model.state_dict())
ema_model.eval()

diffuser = diff.CosSchDiffuser(steps=NOISE_STEPS, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=get_cosine_lambda(initial_lr=LR, final_lr=FINAL_LR, epochs=EPOCHS, warmup_epoch=WARMUP_EPOCHS),
)

num_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'model: {model.__class__.__name__}')
print(f'trainable parameters: {num_params:,}')
print(f'optimizer: AdamW(lr={LR})')
print(f'diffuser: {diffuser.name}, steps={diffuser.steps}')


def train_step(model: torch.nn.Module, batch, diffuser: diff.Diffuser, device: torch.device) -> torch.Tensor:
    condition, targets, _diffuse, _meta = batch
    condition = condition.to(device)
    targets = targets.to(device)

    batch_size = condition.shape[0]
    t = torch.randint(0, diffuser.steps, (batch_size,), dtype=torch.long, device=device)
    noise = torch.randn_like(targets)
    noisy_xt = diffuser.forward_diffusion(targets, t, noise)
    prediction = model(noisy_xt, t, condition)
    return F.mse_loss(prediction, noise)


def evaluate_noise_loss(model: torch.nn.Module, loader, diffuser: diff.Diffuser, device: torch.device, max_batches: int | None = None) -> float:
    was_training = model.training
    model.eval()
    losses = []
    with torch.no_grad():
        for idx, batch in enumerate(loader):
            if max_batches is not None and idx >= max_batches:
                break
            losses.append(float(train_step(model, batch, diffuser, device).item()))
    if was_training:
        model.train()
    return float(np.mean(losses)) if losses else float('nan')


def run_eval(
    model: torch.nn.Module,
    loader,
    diffuser: diff.Diffuser,
    fixed_condition: torch.Tensor,
    fixed_target: torch.Tensor,
    fixed_diffuse: torch.Tensor,
    fixed_noise: torch.Tensor,
    device: torch.device,
    epoch: int,
    max_batches: int | None = None,
):
    val_noise_loss = evaluate_noise_loss(model, loader, diffuser, device, max_batches=max_batches)

    with torch.no_grad():
        pure_prediction = ddpm_sample_safe(
            model,
            diffuser,
            fixed_condition,
            initial_noise=fixed_noise,
            show_progress=False,
        )
        restore_prediction = ddpm_restore_safe(
            model,
            diffuser,
            fixed_condition,
            fixed_condition,
            start_step=RESTORE_START_STEP,
            initial_noise=fixed_noise,
            show_progress=False,
        )

    target = fixed_target.squeeze(0)
    condition = fixed_condition.squeeze(0)
    true_diffuse = fixed_diffuse.squeeze(0)
    pure_metrics = metric_bundle(pure_prediction.squeeze(0), target, true_diffuse, condition)
    restore_metrics = metric_bundle(restore_prediction.squeeze(0), target, true_diffuse, condition)

    plot_prediction(fixed_condition, fixed_target, pure_prediction, restore_prediction, fixed_diffuse, epoch=epoch)
    return {
        'epoch': epoch,
        'val_noise_loss': val_noise_loss,
        'pure_target_mse': pure_metrics['target_mse'],
        'pure_target_mae': pure_metrics['target_mae'],
        'pure_diffuse_mse': pure_metrics['diffuse_mse'],
        'pure_diffuse_mae': pure_metrics['diffuse_mae'],
        'pure_diffuse_psnr': pure_metrics['diffuse_psnr'],
        'restore_target_mse': restore_metrics['target_mse'],
        'restore_target_mae': restore_metrics['target_mae'],
        'restore_diffuse_mse': restore_metrics['diffuse_mse'],
        'restore_diffuse_mae': restore_metrics['diffuse_mae'],
        'restore_diffuse_psnr': restore_metrics['diffuse_psnr'],
    }


In [ ]:
progress_bar = tqdm(total=EPOCHS * len(train_loader), desc=f'Training [{TASK_MODE}]', dynamic_ncols=True)
train_loss_history = []
eval_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = train_step(model, batch, diffuser, DEVICE)
        loss.backward()
        optimizer.step()
        update_ema(ema_model, model, decay=EMA_DECAY)

        epoch_loss += float(loss.item())
        progress_bar.update(1)
        progress_bar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')

    epoch_loss /= len(train_loader)
    train_loss_history.append(epoch_loss)
    scheduler.step()

    print(f'Epoch {epoch:4d} | train noise loss {epoch_loss:.6f}')

    if epoch % CHECKPOINT_EVERY == 0:
        checkpoint_save(model, optimizer, epoch_loss, epoch, TRAINER_SAVE_PATH)

    if epoch % EVAL_EVERY == 0:
        metrics = run_eval(
            ema_model,
            val_loader,
            diffuser,
            fixed_val_condition,
            fixed_val_target,
            fixed_val_diffuse,
            fixed_eval_noise,
            DEVICE,
            epoch,
            max_batches=VAL_BATCHES,
        )
        eval_history.append(metrics)
        print(
            f"Eval {epoch:4d} | val noise loss {metrics['val_noise_loss']:.6f} | "
            f"pure diffuse mse {metrics['pure_diffuse_mse']:.6f} | pure diffuse psnr {metrics['pure_diffuse_psnr']:.4f} dB | "
            f"restore diffuse mse {metrics['restore_diffuse_mse']:.6f} | restore diffuse psnr {metrics['restore_diffuse_psnr']:.4f} dB"
        )

progress_bar.close()

torch.save(
    {
        'epoch': EPOCHS,
        'notebook_group': NOTEBOOK_GROUP,
        'task_mode': TASK_MODE,
        'condition_source': CONDITION_SOURCE,
        'target_source': TARGET_SOURCE,
        'restore_prior_source': 'glossy',
        'restore_start_step': RESTORE_START_STEP,
        'image_size': IMAGE_SIZE,
        'noise_steps': NOISE_STEPS,
        'depth': DEPTH,
        'diffuser_module': 'Diffuser',
        'sampling_mode': 'ddpm_safe_e_with_restore_eval',
        'model_state_dict': model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss_history': train_loss_history,
        'eval_history': eval_history,
    },
    FINAL_MODEL_PATH,
)
print(f'Final model saved to {FINAL_MODEL_PATH}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_loss_history)
axes[0].set_title('Train Noise Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].grid(True, alpha=0.3)

if eval_history:
    eval_epochs = [item['epoch'] for item in eval_history]
    axes[1].plot(eval_epochs, [item['diffuse_mse'] for item in eval_history], label='diffuse mse')
    axes[1].plot(eval_epochs, [item['target_mse'] for item in eval_history], label='target mse')
    axes[1].set_title('Validation Curves')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MSE')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].axis('off')

plt.tight_layout()
plt.show()
